# AI News Aggregator

## Final Project
   - ✅ Pahse 1 : Collect news from multiple sources 
   - ✅ Pahse 2 : Summarize articles using AI
   - ✅ Pahse 3 : Detect and remove duplicate news
   - ✅ Pahse 4 : Categorize articles by topic
   - ✅ Pahse 5 : Use an AI agent to rank news based on relevance and importance
   - ✅ Pahse 6 : Store the processed data in an SQL database
   - ✅ Pahse 7 : Generate and send a newsletter

### Project Structure
```
ai-news-aggregator/
│
├── app/
│
│   ├── agents/
│   │      news_agent.py
│   │
│   ├── collectors/
│   │      youtube.py
│   │      rss.py
│   │      blogs.py
│   │      twitter.py
│   │
│   ├── llm/
│   │      provider.py
│   │
│   ├── models/
│   │      article.py
│   │
│   ├── database/
│   │      database.py
│   │
│   ├── services/
│   │      summarizer.py
│   │      classifier.py
│   │      newsletter.py
│   │
│   ├── api/
│   │      routes.py
│   │
│   ├── workers/
│   │      celery_worker.py
│   │
│   ├── scheduler/
│   │      jobs.py
│   │
│   ├── config.py
│   └── main.py
│
├── docker-compose.yml
├── Dockerfile
├── requirements.txt
├── .env
└── README.md

```


#### create project structure
``` python 
# 1. Create the project folder and all subdirectories
mkdir -p app/{agents,collectors,llm,models,database,services,api,workers,scheduler}

# 2. Create all the empty Python files
touch app/agents/news_agent.py \
      app/collectors/{youtube,rss,blogs,twitter}.py \
      app/llm/provider.py \
      app/models/article.py \
      app/database/database.py \
      app/services/{summarizer,classifier,newsletter}.py \
      app/api/routes.py \
      app/workers/celery_worker.py \
      app/scheduler/jobs.py \
      app/{config,main}.py

# 3. Create root files
touch   {docker-compose.yml,Dockerfile,requirements.txt,.env,README.md}
```
-----------------------------------------

##  ✅ Phase 1 — Collect News

Build a system that can collect news from multiple sources and return it in a single standardized format.

```
RSS Feed
        \
YouTube -----> Collectors -----> Standard Article Object
        /
Blogs
```

At the end of Phase 1, running the application should produce something like:

```python
[
    {
        "title": "...",
        "content": "...",
        "url": "...",
        "published_at": "...",
        "source": "OpenAI Blog"
    },
    ...
]
```
### ⭐ Steps :
1. I will use these 3 sources:

| Source       | RSS                                                                              |
| ------------ | -------------------------------------------------------------------------------- |
| OpenAI       | [https://openai.com/news/rss.xml](https://openai.com/news/rss.xml)               |
| Anthropic    | [https://www.anthropic.com/news/rss.xml](https://www.anthropic.com/news/rss.xml) |
| Hugging Face | [https://huggingface.co/blog/feed.xml](https://huggingface.co/blog/feed.xml)     |

2. Normalize the sources
Different sources expose different fields.we should normalize them into one structure.
The model should contain:
- title
- summary
- url
- published_at
- source

Every collector must return this exact model, regardless of where the article came from.

3. Create Collectors
- blogs
- rss
- youtube
 
and each collector should return the same model.
```
Fetch RSS
↓
Read entries
↓
Convert entries to Article objects
↓
Return a list of articles
```

4. Create a Collector Interface
Every collector should expose a collect() method.

5. Orchestrator
main.py should not know how RSS works. Instead, it simply coordinates collectors:
```
OpenAI Collector
        │
Anthropic Collector
        │
HF Collector
        │
        ▼
Merge results
        ▼
Print
``` 
This separation is important. main.py orchestrates; collectors implement source-specific logic.

### Folder Structure After Phase 1
```
ai-news-aggregator/
│
├── app/
│   ├── collectors/
│   │   ├── rss.py
│   │   ├── blogs.py
│   │   ├── twitter.py
│   │   └── youtube.py
│   │
│   ├── models/
│   │   └── article.py
│   │
│   ├── config.py
│   └── main.py
│
├── requirements.txt
├── .env
└── README.md
```
----------------

###  ✔ test Phase 1

In [ ]:
from app.config import RSS_FEEDS
from app.collectors.rss import RSSCollector

for source_name, url in RSS_FEEDS.items():
    collector = RSSCollector(source_name, url)
    articles = collector.collect()
    print(articles)


##  ✅ Phase 2 : AI Summarization
where the project starts becoming an AI Engineering project rather than just a data collection project.

- ✔ Build LLM Provider (llm/provider.py)
- ✔ Add Prompt Engineering
- ✔ Build Summarizer Service (services/summarizer.py)
- ✔ Update RSS Collector


###  ✔ test Phase 2

In [ ]:
from app.collectors.rss import RSSCollector
from app.config import RSS_FEEDS
from app.services.summarizer import Summarizer

summarizer = Summarizer()

for source_name, url in RSS_FEEDS.items():

    collector = RSSCollector(source_name, url)

    articles = collector.collect()

    for article in articles:

        article = summarizer.summarize(article)

        print("=" * 80)
        print(article.title)
        print()
        print(article.summary)

##  ✅Phase 2 — Store
Without a database:

You fetch news → process → lose everything

With PostgreSQL:

You keep all articles over time
You can query:
“what was published yesterday?”
“what topics are trending this week?”

PostgreSQL table
``` SQL
CREATE TABLE articles (
    id SERIAL PRIMARY KEY,
    title VARCHAR(500) NOT NULL,
    summary TEXT  NULL,
    url VARCHAR(500) NOT NULL,
    published_at TIMESTAMP  NULL,
    source VARCHAR(100) NOT NULL
);
```



In [ ]:
from app.database.database import insert_articles
from app.config import DATABASE_URL
from sqlalchemy import create_engine

engine = create_engine(DATABASE_URL)
#insert_articles(engine, articles)